# Phase 4 - Self-play GO/NO-GO probe (Colab, L4)

**One open question:** can self-play **beat a strong, deck-MATCHED rule agent**? That is
the load-bearing assumption of the whole plan (the intermediate runs only proved it beats
`random`). Everything already settled - no-collapse demo, throughput sweep, recipe-v2
stability, the option-rank A/B - is parked in `colab_archive_phase0_3.ipynb` so we don't
waste L4 cycles re-answering it.

**Design (deck-controlled, held-out yardstick).** Train **Archaludon** (Agent A) on the
rule agent's *exact* 60-card list (`agent/kaggle_agents/archaludon_deck.csv`) against the
pool with **`kaggle:archaludon` removed** (`mixed_pool_heldout_archaludon.json`), then eval
vs that same hand-coded agent it never trained against. Same deck both sides -> the only
variable is learned-policy vs hand-rules. `mirror` = A/A null (must stay ~50% = unbiased
eval); `random` = does-it-still-learn floor; **`kaggle:archaludon` = the go/no-go number.**

**Two runs, in order:**
1. **`small` probe** - the known-stable recipe (held entropy across 4 prior runs). The
   clean thesis read, isolated from net-size instability.
2. **`medium`-stability variant** - tests whether a bigger net trains *stably* with the
   orbit-wars-informed anti-collapse recipe (the first `medium` run collapsed entropy to 0
   by it8; see the cell for the fix and why).

**Read the `kaggle:archaludon` slope** (not one point): rising & crossing ~50% with
`mirror`~50% -> **GO** (fan out to A+B at full budget); flat in the teens -> **NO-GO**,
escalate (bigger/longer, or BC warm-start + exploiters - see `orbit-wars-selfplay-lessons`).

Runtime -> L4 GPU, run top to bottom. Checkpoints/logs persist to Drive (`--out`), so runs
resume if the session drops; Ctrl-stop once a curve is unambiguous.

In [ ]:
# Mount Drive (artifact persistence), then clone (or update) the repo and cd in.
import os

from google.colab import drive
drive.mount('/content/drive')

# Everything the runs produce is written under here so it survives the session and
# can be pulled back with scripts/download_artifacts.py. Keep this name in sync with
# DRIVE_BASE in that script ('ptcg_outputs').
DRIVE_OUTPUT = '/content/drive/MyDrive/ptcg_outputs'
for sub in ['', '/logs', '/checkpoints_sp_small', '/ablation_sp']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = "https://github.com/oshbocker/pokemon_tcg_ai_battle.git"
REPO_DIR = "/content/pokemon_tcg_ai_battle"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

assert os.path.isfile("agent/cg/libcg.so"), (
    "agent/cg/libcg.so missing - push it to the repo (see .gitignore exception)."
)
import torch
print("repo:", os.getcwd(), "| Drive:", DRIVE_OUTPUT)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Hardware topology - record this alongside the results.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "(no GPU runtime selected)"
!nproc
!lscpu | grep -E "^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Model name"
!free -h | head -2


## 1. `small` probe - clean go/no-go on the thesis

The `small` (5.7M) recipe held entropy without collapse across 4 prior L4 runs, so this
isolates the thesis question from net-size instability. `--workers 48` (throughput sweep
peak), `--gate-threshold 0.6` (raised from 0.55 toward the orbit-wars 70-80% bar).

In [ ]:
# -- GO/NO-GO probe (small, known-stable recipe) --
PROBE_OUT = f'{DRIVE_OUTPUT}/probe_archaludon_small'
LOG = '/content/colab_probe_archaludon_small.txt'
!python scripts/train_selfplay.py \
    --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_heldout_archaludon.json \
    --size small --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 10 \
    --iters 200 --games-per-iter 128 --epochs 2 --minibatch 256 \
    --lr 3e-4 --lr-final 5e-5 --target-entropy 0.1 --target-kl 1.5 \
    --gate --gate-every 10 --gate-games 200 --gate-threshold 0.6 \
    --eval-every 10 --eval-games 200 \
    --eval-opponents mirror,random,kaggle:archaludon \
    --device cuda --seed 0 \
    --out {PROBE_OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## 2. `medium`-stability variant - anti-collapse recipe (orbit-wars-informed)

The first `medium` (15.8M) run **collapsed**: entropy hit 0 by it8 and the ratcheting
controller couldn't recover (its docstring predicts exactly this - once ent=0 the entropy
gradient vanishes). Cause: LR 3e-4 + tiny batch drove the bigger net deterministic before
the controller could react. Fixes, straight from the Orbit Wars self-play write-ups
(Gerar Del Toro: 1e-4 LR, grad-clip 0.5, **1 PPO epoch**; IsaiahP: *"bigger batch + lower
LR"* to break plateaus; higher promotion bar 70-80%):

| knob | was | now | why |
|---|---|---|---|
| `--lr` | 3e-4 | **1e-4** | smaller steps; no early entropy dive |
| `--games-per-iter` | 128 | **256** | bigger batch = lower-variance grads |
| `--epochs` | 2 | **1** | less per-batch overfit/drift |
| `--ent-coef` (floor) | 0.02 | **0.04** | keep the bonus live through the early phase |
| `--target-entropy` | 0.1 | **0.15** | controller reacts sooner |
| `--gate-threshold` | 0.55 | **0.65** | don't promote noisy/weak references |

(grad-norm clip 0.5 + advantage normalization are already on by default in `ppo.py`.)
Watch `ent`: it should hold above ~0.05, not crater to 0 like last time.

In [ ]:
# -- MEDIUM-stability variant (anti-collapse recipe) --
PROBE_OUT = f'{DRIVE_OUTPUT}/probe_archaludon_medium_v2'
LOG = '/content/colab_probe_archaludon_medium_v2.txt'
!python scripts/train_selfplay.py \
    --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_heldout_archaludon.json \
    --size medium --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 10 \
    --iters 200 --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr 1e-4 --lr-final 1e-5 --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --gate --gate-every 10 --gate-games 200 --gate-threshold 0.65 \
    --eval-every 10 --eval-games 200 \
    --eval-opponents mirror,random,kaggle:archaludon \
    --device cuda --seed 0 \
    --out {PROBE_OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## Results are on Drive — pull them to the laptop

Everything the runs produced lives under `MyDrive/ptcg_outputs/` (checkpoints, eval
CSVs, and the `logs/colab_*.txt` records). Nothing needs to be git-pushed from here.
The cell below just confirms what landed; back on the laptop, fetch it with rclone:

```bash
uv run python scripts/download_artifacts.py --logs                 # colab_*.txt -> rl_research/
uv run python scripts/download_artifacts.py --run ablation_sp      # A/B ckpts + CSVs -> outputs/
uv run python scripts/download_artifacts.py --run checkpoints_sp_small
```

Then paste the A/B table + RECOMMENDATION into `ABLATION_OPTION_RANK.md` and commit
the logs. (One-time rclone `gdrive` remote setup is in CLAUDE.md.)

In [ ]:
# Confirm what persisted to Drive.
!ls -lhR {DRIVE_OUTPUT} | sed -n '1,80p'
